# 🎓 Project Integra — Module 1 Evaluation Demo
**Automated Lecture Video Summarization — Keyframe Detection & Importance Scoring**

This notebook demonstrates the complete Module 1 software pipeline:
1. ResNet-50 visual feature extraction
2. BiLSTM importance scoring model (architecture + forward pass)
3. Annotation data loading and statistics
4. Training pipeline readiness check

## Step 1 — Install dependencies

In [ ]:
!pip install torch torchvision --quiet
!pip install opencv-python-headless numpy matplotlib --quiet
print('✅ Dependencies installed')

## Step 2 — Clone the repository

In [ ]:
import os

REPO_URL = 'https://github.com/rashmiJayawardhana/lecture-video-summarizer.git'
BRANCH   = 'module-1'

!git clone --branch {BRANCH} {REPO_URL} repo
os.chdir('repo')
print('✅ Repository cloned, working directory:', os.getcwd())

## Step 3 — Verify the ResNet-50 feature extractor

In [ ]:
import sys
sys.path.insert(0, '.')

import torch
import numpy as np
from src.module1_importance.feature_extractor import FrameFeatureExtractor

# Use CPU for demo (no GPU needed)
extractor = FrameFeatureExtractor(device='cpu')

# Simulate a single video frame (480x640 RGB)
dummy_frame = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)
features = extractor.extract_from_frame(dummy_frame)

print('=== ResNet-50 Feature Extractor ===')
print(f'  Input  : single video frame  (480 x 640 x 3)')
print(f'  Output : feature vector shape = {features.shape}')
print(f'  Range  : [{features.min():.3f}, {features.max():.3f}]')
print()
print('✅ Feature extractor works correctly')

## Step 4 — Verify the BiLSTM model architecture

In [ ]:
from src.module1_importance.model import VideoImportanceScorer

model = VideoImportanceScorer(hidden_size=512, num_lstm_layers=2, dropout=0.3)

# Simulate a batch: 4 video segments, each 10 frames, each frame = 2048-dim vector
BATCH_SIZE      = 4
FRAMES_PER_SEG  = 10   # 10-second segment at 1 FPS
FEATURE_DIM     = 2048 # ResNet-50 output

dummy_input = torch.randn(BATCH_SIZE, FRAMES_PER_SEG, FEATURE_DIM)
output = model(dummy_input)

total_params = sum(p.numel() for p in model.parameters())

print('=== VideoImportanceScorer (ResNet-50 + BiLSTM) ===')
print(f'  Input  shape : {list(dummy_input.shape)}   (batch x frames x features)')
print(f'  Output shape : {list(output.shape)}        (batch x frames)')
print(f'  Score range  : [{output.min().item():.4f}, {output.max().item():.4f}]  (should be 0-1)')
print(f'  Parameters   : {total_params:,} trainable')
print()
print('  Sample segment scores (segment 1, all 10 frames):')
for i, s in enumerate(output[0].detach().tolist()):
    bar = '█' * int(s * 20)
    print(f'    Frame {i+1:2d}: {s:.4f}  {bar}')
print()
print('✅ BiLSTM model forward pass verified')

## Step 5 — Load and visualise the calibration annotation data

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

with open('module1_annotations.json', encoding='utf-8') as f:
    annotations = json.load(f)

segments    = list(annotations.values())
raw_scores  = [s['raw_score']        for s in segments]
norm_scores = [s['normalized_score'] for s in segments]
timestamps  = [s['timestamp_start']  for s in segments]

print('=== module1_annotations.json (calibration round — LecVideo 045) ===')
print(f'  Total segments annotated : {len(segments)}')
print(f'  Video duration covered   : ~{max(s["timestamp_end"] for s in segments)/60:.1f} minutes')
print(f'  Score range              : {min(raw_scores)} – {max(raw_scores)}')
print(f'  Mean score               : {sum(raw_scores)/len(raw_scores):.2f}')
print()

# Score distribution
from collections import Counter
dist = Counter(raw_scores)
print('  Score distribution:')
for sc in range(11):
    count = dist.get(sc, 0)
    pct   = count / len(raw_scores) * 100
    bar   = '█' * int(pct / 2)
    band  = '(Critical)' if sc >= 8 else ('(Useful)' if sc >= 4 else ('(Low)' if sc >= 1 else '(Filler)'))
    print(f'    {sc:2d} : {bar:<30} {count:3d}  ({pct:.1f}%)  {band}')

In [ ]:
# Plot importance scores across the lecture timeline
fig, axes = plt.subplots(2, 1, figsize=(14, 7))
fig.suptitle('Module 1 — LecVideo 045 Annotation Results (Calibration Round)', fontsize=14, fontweight='bold')

# Timeline plot
colors = []
for s in raw_scores:
    if s >= 8:   colors.append('#e74c3c')   # red — critical
    elif s >= 4: colors.append('#f39c12')   # orange — useful
    elif s >= 1: colors.append('#3498db')   # blue — low value
    else:        colors.append('#95a5a6')   # grey — filler

axes[0].bar([t/60 for t in timestamps], norm_scores, width=10/60, color=colors, alpha=0.85, edgecolor='none')
axes[0].set_xlabel('Lecture time (minutes)')
axes[0].set_ylabel('Importance score (0–1)')
axes[0].set_title('Segment importance scores across lecture timeline')
axes[0].set_ylim(0, 1.1)
patches = [
    mpatches.Patch(color='#e74c3c', label='Critical (8–10)'),
    mpatches.Patch(color='#f39c12', label='Useful (4–7)'),
    mpatches.Patch(color='#3498db', label='Low value (1–3)'),
    mpatches.Patch(color='#95a5a6', label='Filler (0)'),
]
axes[0].legend(handles=patches, loc='upper right')

# Score distribution histogram
score_labels = [str(i) for i in range(11)]
score_counts = [dist.get(i, 0) for i in range(11)]
bar_colors   = ['#95a5a6'] + ['#3498db']*3 + ['#f39c12']*4 + ['#e74c3c']*3
axes[1].bar(score_labels, score_counts, color=bar_colors, edgecolor='white', linewidth=0.5)
axes[1].set_xlabel('Raw score (0–10)')
axes[1].set_ylabel('Number of segments')
axes[1].set_title('Distribution of annotation scores')
for i, v in enumerate(score_counts):
    if v > 0:
        axes[1].text(i, v + 0.5, str(v), ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('module1_calibration_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved to module1_calibration_chart.png')

## Step 6 — Show the training pipeline is ready

In [ ]:
from src.module1_importance.train import LectureFeatureDataset
import pathlib

# Demonstrate dataset class loads and parses the annotations correctly
# (No .npy feature files yet — this shows what the loader does)
print('=== LectureFeatureDataset — Training pipeline readiness ===')
print()

class DryRunDataset(LectureFeatureDataset):
    """Subclass that skips the .npy file requirement for the dry-run demo."""
    def __init__(self, annotations_json, split):
        import json
        self.features_dir = pathlib.Path('data/processed/features')
        self.split = split
        self.sequence_length = 10
        self.video_cache = {}
        self.samples = []
        with open(annotations_json, encoding='utf-8') as f:
            all_annotations = json.load(f)
        for seg_id, data in all_annotations.items():
            if data.get('skipped', False) or data.get('normalized_score') is None:
                continue
            video_name = data['video_id']
            try:
                vid_num = int(video_name.split()[1])
            except (IndexError, ValueError):
                continue
            if split == 'train' and vid_num > 45: continue
            if split == 'val'   and (vid_num <= 45 or vid_num > 50): continue
            if split == 'test'  and vid_num <= 50: continue
            self.samples.append(data)

train_ds = DryRunDataset('module1_annotations.json', 'train')
val_ds   = DryRunDataset('module1_annotations.json', 'val')
test_ds  = DryRunDataset('module1_annotations.json', 'test')

print(f'  Calibration video (LecVideo 045) → in TRAIN split')
print(f'  Train segments loaded   : {len(train_ds)}')
print(f'  Val   segments loaded   : {len(val_ds)}')
print(f'  Test  segments loaded   : {len(test_ds)}')
print()
print('  Sample training record:')
s = train_ds.samples[44]
print(f'    segment_id      : {s["segment_id"]}')
print(f'    timestamp       : {s["timestamp_start"]}s – {s["timestamp_end"]}s')
print(f'    raw_score       : {s["raw_score"]} / 10')
print(f'    normalized_score: {s["normalized_score"]} (used as training label)')
print(f'    annotator       : {s["annotator"]}')
print()
print('  ⚙️  Next step: download training videos → run extract_all_features.py → run train.py')
print()
print('✅ Training pipeline data loader verified')

## Step 7 — Inter-module JSON schema sample

In [ ]:
import json

# Show what Module 1's output JSON will look like when inference is running
sample_output = {
    "seg_001": {
        "segment_id":      "LecVideo_001__seg_0044",
        "timestamp_start": 440.0,
        "timestamp_end":   450.0,
        "score_V":         0.90
    },
    "seg_002": {
        "segment_id":      "LecVideo_001__seg_0045",
        "timestamp_start": 450.0,
        "timestamp_end":   460.0,
        "score_V":         0.10
    },
    "seg_003": {
        "segment_id":      "LecVideo_001__seg_0066",
        "timestamp_start": 660.0,
        "timestamp_end":   670.0,
        "score_V":         0.72
    }
}

print('=== Module 1 → Module 4 inter-module JSON schema ===')
print(json.dumps(sample_output, indent=2))
print()
print('  score_V is the visual importance score produced by the BiLSTM.')
print('  Module 4 fuses this with score_T (Module 2) and score_L (Module 3)')
print('  using: S = w1*V + w2*T + w3*L  to select the best segments.')

## ✅ Summary — What Module 1 has delivered

| Component | Status |
|-----------|--------|
| ResNet-50 feature extractor (`feature_extractor.py`) | ✅ Implemented & tested |
| BiLSTM model architecture (`model.py`) | ✅ Implemented & tested |
| Training loop with MSE loss (`train.py`) | ✅ Implemented & ready |
| Batch feature extraction script (`extract_all_features.py`) | ✅ Ready to run |
| Annotation tool with GUI (`annotate_module1.py`) | ✅ Built & deployed to all 4 annotators |
| Calibration annotation — LecVideo 045 | ✅ 289 segments, all 4 members |
| Merge pipeline (`--merge`) | ✅ Tested — produced `module1_annotations.json` |
| Calibration alignment | ✅ 74% agreement (180 exact + 36 diff≤1) |
| Inter-module JSON schema | ✅ Agreed and frozen |
| **Pending** | Download training videos → extract features → train model |